# Candidate SSE Mixing Association: Observed Entropies

Runs the node-level mixing regression using observed normalised entropy columns (`*_entropy_obs`) instead of null-model z-scores. This notebook saves only the mixing-model outputs.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection.lib.association_pipeline import (  # noqa: E402
    OBSERVED_MIXING_FEATURES,
    default_model_sets,
    run_association_pipeline,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## Model Specification

In [2]:
RESULT_DIR = PROJECT_ROOT / "sse_detection" / "observed_entropy_sensitivity"
MODEL_METHOD = "firth_glm"
MIXING_REFERENCE = "per 1-unit observed normalised entropy"

MIXING_MODEL_SETS = default_model_sets(
    variant_adjuster="clade",
    window_adjustment="fixed_effects",
)

OBSERVED_MIXING_FEATURES, MIXING_MODEL_SETS

(['sex_entropy_obs',
  'age_entropy_obs',
  'simd_entropy_obs',
  'urban_rural_entropy_obs',
  'health_board_entropy_obs'],
 {'primary': ['C(window_idx)', 'C(clade)'],
  'expanded': ['C(window_idx)',
   'C(clade)',
   'z_dz_cum_prop_sequenced',
   'z_dz_cum_incidence_per_capita',
   'z_dz_7d_test_positivity',
   'z_log1p_dz_cum_positive_tests']})

## Fit and Save

In [3]:
result = run_association_pipeline(
    project_root=PROJECT_ROOT,
    result_dir=RESULT_DIR,
    model_method=MODEL_METHOD,
    variant_adjuster="clade",
    window_adjustment="fixed_effects",
    mixing_model_sets=MIXING_MODEL_SETS,
    mixing_features=OBSERVED_MIXING_FEATURES,
    mixing_reference=MIXING_REFERENCE,
    run_composition=False,
    run_mixing=True,
)

print(f"Results saved to: {result['result_dir']}")
display(result["cluster_diagnostics"])
{name: len(table) for name, table in result["summary_tables"].items()}

Fitted mixing__primary__single__sex_entropy_obs: 13,058 nodes
Fitted mixing__primary__single__age_entropy_obs: 13,058 nodes
Fitted mixing__primary__single__simd_entropy_obs: 13,058 nodes
Fitted mixing__primary__single__urban_rural_entropy_obs: 13,058 nodes
Fitted mixing__primary__single__health_board_entropy_obs: 13,058 nodes
Fitted mixing__primary__joint: 13,058 nodes
Fitted mixing__expanded__single__sex_entropy_obs: 13,058 nodes
Fitted mixing__expanded__single__age_entropy_obs: 13,058 nodes
Fitted mixing__expanded__single__simd_entropy_obs: 13,058 nodes
Fitted mixing__expanded__single__urban_rural_entropy_obs: 13,058 nodes
Fitted mixing__expanded__single__health_board_entropy_obs: 13,058 nodes
Fitted mixing__expanded__joint: 13,058 nodes
saved mixing_wald.csv: 20 rows
saved mixing_odds_ratios.csv: 20 rows
saved mixing_fit_stats.csv: 12 rows
Results saved to: /Users/ydnkka/Desktop/PhD Project/projects/scotland/sse_detection/observed_entropy_sensitivity


,cluster_col,n_rows,n_clusters,min_rows_per_cluster,median_rows_per_cluster,outcome_positive_clusters,outcome_varying_clusters,analysis_frame
0,cluster_id,13059,13059,1,1.0,6907,0,node_mixing


{'mixing_wald.csv': 20,
 'mixing_odds_ratios.csv': 20,
 'mixing_fit_stats.csv': 12}

In [4]:
if not result["failures"].empty:
    display(result["failures"])
else:
    print("No model failures recorded.")

No model failures recorded.
